[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/factor-momentum-colab-v1.1/Momentum_systematic_strategies/02_factor_momentum/strategy_candidates/xle_systematic_momentum/notebooks/02_pre_paper_review.ipynb)

**Notebook lista para reproducirse en Google Colab — ejecución completa verificada en noviembre de 2026.**

**Haz clic en «Abrir en Colab» y selecciona «Entorno de ejecución → Ejecutar todas» (Run all). La notebook prepara automáticamente las dependencias compatibles, descarga y verifica los siete archivos Parquet Gold desde una revisión inmutable de GitHub y, cuando corresponde, recupera y valida las notebooks de referencia mediante SHA-256.**

**No necesitas claves de API, Google Drive, archivos locales ni ejecutar notebooks anteriores. Todos los cálculos se realizan con datos públicos y código de investigación versionado.**

In [ ]:
# Bootstrap P1 para Colab. Las fuentes analíticas se descargan sólo desde P0.
from datetime import date
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen
import json
import os
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    # Colab puede cargar NumPy antes de esta celda. Reinstalarlo no sustituye el módulo en memoria.
    SUPPORTED_NUMPY_VERSIONS = ("2.4.6", "2.5.4")
    loaded_numpy = sys.modules.get("numpy")
    try:
        numpy_distribution = version("numpy")
    except PackageNotFoundError:
        numpy_distribution = None
    numpy_runtime = getattr(loaded_numpy, "__version__", numpy_distribution)
    if numpy_runtime not in SUPPORTED_NUMPY_VERSIONS:
        raise RuntimeError(f"NumPy {numpy_runtime!r} no pertenece al contrato reproducible {SUPPORTED_NUMPY_VERSIONS}. Reinicia el entorno Colab limpio.")
    REQUIRED_PACKAGES = {"polars": "1.44.2", "plotly": "6.2.0", "nbformat": "5.10.4"}
    installed = {}
    for package, required in REQUIRED_PACKAGES.items():
        loaded = sys.modules.get(package)
        if loaded is not None and getattr(loaded, "__version__", None) != required:
            raise RuntimeError(f"{package} ya está cargado con una versión incompatible. Reinicia el entorno Colab limpio.")
        try:
            installed[package] = version(package)
        except PackageNotFoundError:
            installed[package] = None
    if any(installed[name] != required for name, required in REQUIRED_PACKAGES.items()):
        command = [sys.executable, "-m", "pip", "install", "--quiet"]
        command.extend(f"{package}=={required}" for package, required in REQUIRED_PACKAGES.items())
        subprocess.check_call(command)
    import numpy as np
    import nbformat
    import plotly
    import polars as pl
    if (np.__version__ not in SUPPORTED_NUMPY_VERSIONS or pl.__version__ != "1.44.2" or
            plotly.__version__ != "6.2.0" or nbformat.__version__ != "5.10.4"):
        raise RuntimeError("Las versiones instaladas no coinciden con el entorno reproducible fijado.")

    GOLD_COMMIT = "a8ab4d6bf04184551a783925eec1a94db0a31dd9"
    P0_COMMIT = "af9a5f44412cc940fc655af92f74669fe341e84b"
    INPUTS = {
        "SPY": "b0767b45c2d67dbe8d99e11e1603682fd28468c31de65d037a4c0a50fefa67e9",
        "QQQ": "69760c426b11238aca1f1200f1c44933938954115d944db3b5a3df77618475fc",
        "IWM": "7913707e8a33b60780a7fe93617eb8114ef71b0e66e6a3b0b4dbab752cf09523",
        "IWD": "d73576d8c83dcbe57392174cb341061321ab963cd4adbdd063f455342caea0e0",
        "IWF": "d0c4165277bc9167421dde15ef9ede395bc35d259326de22913a76ef5792314e",
        "XLE": "f98e7468b5aee758298229a14443818ea0148bc46b26042a02abe1dee9af7a5a",
        "XLF": "52612673ab44bb3e222d5ac208bd3b9dfd0871c2259ff9fb472c59d902f170e0",
    }
    P0_NOTEBOOKS = {
        "signal": ("02_factor_momentum/notebooks/02_factor_momentum_signal_test.ipynb", "5c43aa1b60930b4f16d6e60e13f818ca1a3ae23569ec64a4cf96adc1b0c97c5e"),
        "execution": ("02_factor_momentum/strategy_candidates/xle_systematic_momentum/notebooks/01_execution_feasibility.ipynb", "ab5a9d433423095a038ecdd2516eee7cda449cec3b572a2f78a08ca425ccf9a0"),
    }
    REPO = "https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes"
    PROJECT_ROOT = Path("/content/quant-research-notes/Momentum_systematic_strategies")
    GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
    SOURCE_ROOT = PROJECT_ROOT / "_p0_sources"

    def download_verified(url, destination, expected_hash):
        if destination.exists():
            if sha256(destination.read_bytes()).hexdigest() != expected_hash:
                raise RuntimeError(f"{destination.name}: existe un archivo con SHA-256 distinto.")
            return destination
        try:
            with urlopen(url, timeout=60) as response:
                if response.status != 200:
                    raise RuntimeError(f"HTTP {response.status} al descargar {url}")
                payload = response.read()
        except (HTTPError, URLError, TimeoutError) as error:
            raise RuntimeError(f"No fue posible descargar {url}.") from error
        if sha256(payload).hexdigest() != expected_hash:
            raise RuntimeError(f"SHA-256 remoto inesperado para {destination.name}.")
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(destination.suffix + ".download")
        temporary.write_bytes(payload)
        temporary.replace(destination)
        return destination

    for symbol, expected_hash in INPUTS.items():
        gold_url = f"{REPO}/{GOLD_COMMIT}/Momentum_systematic_strategies/data/gold/daily_series/etf/{symbol}.parquet"
        download_verified(gold_url, GOLD_ROOT / f"{symbol}.parquet", expected_hash)
    calendars = []
    for symbol in INPUTS:
        frame = pl.read_parquet(GOLD_ROOT / f"{symbol}.parquet").with_columns(pl.col("session_date").cast(pl.Date))
        scoped = frame.filter(pl.col("session_date").is_between(date(2018, 1, 2), date(2026, 11, 17), closed="both"))
        if (set(("session_date", "adjusted_close")) - set(frame.columns) or scoped.height != 2232 or
                scoped.get_column("session_date").null_count() or scoped.get_column("adjusted_close").null_count() or
                scoped.get_column("session_date").is_duplicated().any() or (scoped.get_column("adjusted_close") <= 0).any()):
            raise RuntimeError(f"Gold inválido para {symbol}.")
        calendars.append(set(scoped.get_column("session_date").to_list()))
    if len(set.intersection(*calendars)) != 2232:
        raise RuntimeError("La intersección Gold no coincide con las 2,232 sesiones esperadas.")

    def materialize_without_bootstrap(source, destination, omitted_id):
        original = json.loads(source.read_text(encoding="utf-8"))
        omitted = [cell for cell in original["cells"] if cell.get("id") == omitted_id]
        if len(omitted) != 1 or omitted[0].get("cell_type") != "code":
            raise RuntimeError(f"No se encontró el bootstrap único {omitted_id}.")
        retained = [cell for cell in original["cells"] if cell.get("id") != omitted_id]
        materialized = dict(original)
        materialized["cells"] = retained
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(json.dumps(materialized, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
        reopened = json.loads(destination.read_text(encoding="utf-8"))
        if reopened["cells"] != retained:
            raise RuntimeError(f"Las celdas retenidas de {source.name} no se conservaron exactamente.")
        return {"source_sha256": sha256(source.read_bytes()).hexdigest(), "omitted": omitted_id, "retained_cells": len(retained)}

    p0_sources = {}
    for name, (relative, expected_hash) in P0_NOTEBOOKS.items():
        source = SOURCE_ROOT / f"{name}-p0.ipynb"
        p0_sources[name] = download_verified(f"{REPO}/{P0_COMMIT}/Momentum_systematic_strategies/{relative}", source, expected_hash)
    signal_target = PROJECT_ROOT / "02_factor_momentum" / "notebooks" / "02_factor_momentum_signal_test.ipynb"
    execution_target = PROJECT_ROOT / "02_factor_momentum" / "strategy_candidates" / "xle_systematic_momentum" / "notebooks" / "01_execution_feasibility.ipynb"
    execution_dependency = execution_target.with_name("01_execution_feasibility_colab_dependency.ipynb")
    dependency_audit = {
        "signal": materialize_without_bootstrap(p0_sources["signal"], signal_target, "colab-bootstrap-signal"),
        "execution": materialize_without_bootstrap(p0_sources["execution"], execution_dependency, "colab-bootstrap-execution"),
    }
    execution_target.parent.mkdir(parents=True, exist_ok=True)
    execution_target.write_bytes(p0_sources["execution"].read_bytes())
    if sha256(execution_target.read_bytes()).hexdigest() != P0_NOTEBOOKS["execution"][1]:
        raise RuntimeError("La copia P0 de Execution no conserva su SHA-256.")
    COLAB_DEPENDENCY_AUDIT = {"gold_commit": GOLD_COMMIT, "p0_commit": P0_COMMIT, "omitted_infrastructure": dependency_audit}
    os.chdir(PROJECT_ROOT)
    print(f"Colab P1 listo | Gold {GOLD_COMMIT} | código P0 {P0_COMMIT}")
    print(f"Infraestructura omitida sólo en copias de dependencia: {dependency_audit}")
else:
    print("Ejecución local detectada: P1 no descarga ni materializa dependencias remotas.")


# 02 — XLE Systematic Momentum: revisión previa a paper trading

> Revisión exploratoria previa a paper trading. La señal, los factores y sus parámetros se mantienen congelados; este cuaderno no transmite órdenes.

## <b>Reproducción y sensibilidad a fricciones</b>

La notebook de ejecución se carga en memoria hasta su tabla de resumen. Esto conserva la identidad de señal ya auditada y permite variar solamente el supuesto ilustrativo de slippage.

In [ ]:
from pathlib import Path
import json

import numpy as np
import polars as pl

def candidate_root(start: Path) -> Path:
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies' / '02_factor_momentum' / 'strategy_candidates' / 'xle_systematic_momentum'
        if (direct / 'notebooks' / '01_execution_feasibility.ipynb').exists():
            return direct
        candidate = parent / 'strategy_candidates' / 'xle_systematic_momentum'
        if (candidate / 'notebooks' / '01_execution_feasibility.ipynb').exists():
            return candidate
        if (parent / 'notebooks' / '01_execution_feasibility.ipynb').exists():
            return parent
    raise FileNotFoundError('No se encontró la candidata XLE.')

CANDIDATE_ROOT = candidate_root(Path.cwd())
dependency_path = CANDIDATE_ROOT / 'notebooks' / '01_execution_feasibility_colab_dependency.ipynb'
source_path = dependency_path if dependency_path.exists() else CANDIDATE_ROOT / 'notebooks' / '01_execution_feasibility.ipynb'
source = json.loads(source_path.read_text(encoding='utf-8'))
state = {'__name__': '__pre_paper_review__'}
for cell in source['cells']:
    if cell['cell_type'] == 'code':
        code = ''.join(cell['source'])
        exec(compile(code, str(source_path), 'exec'), state)
        if 'execution_summary =' in code:
            break

schedule = state['execution_schedule']
run_execution = state['run_execution']
scenarios = state['SCENARIOS']
strategies = state['STRATEGIES']
initial_capital = state['INITIAL_CAPITAL']
xle_ohlc = state['xle_ohlc']
state['signal_reproduction_audit']

In [ ]:
def execution_metrics(history, trades):
    nav = history.get_column('nav').to_numpy()
    nav_points = np.r_[initial_capital, nav]
    log_returns = np.diff(np.log(nav_points))
    drawdown = nav_points / np.maximum.accumulate(nav_points) - 1
    return {
        'final_capital': nav[-1],
        'cagr_pct': ((nav[-1] / initial_capital) ** (252 / len(log_returns)) - 1) * 100,
        'annualized_volatility_pct': log_returns.std(ddof=1) * np.sqrt(252) * 100,
        'sharpe': log_returns.mean() / log_returns.std(ddof=1) * np.sqrt(252),
        'max_drawdown_pct': drawdown.min() * 100,
        'operations': trades.height,
        'commission_usd': trades.get_column('commission').sum() if trades.height else 0.0,
        'slippage_usd': trades.get_column('slippage_cost').sum() if trades.height else 0.0,
    }

friction_rows = []
for bps in (0.0, 5.0, 10.0, 20.0):
    scenario = f'Review fixed + {bps:g} bps'
    scenarios[scenario] = {'commission': True, 'slippage_bps': bps}
    history, trades = run_execution(schedule, 'Systematic momentum', scenario)
    friction_rows.append({'slippage_bps': bps, **execution_metrics(history, trades)})

friction_review = pl.DataFrame(friction_rows).with_columns(pl.all().exclude('operations').round(3))
friction_review

## <b>Auditoría temporal y contable</b>

> La sensibilidad de un día conserva cada fecha de decisión y desplaza únicamente la apertura de ejecución a la siguiente sesión disponible.

In [ ]:
base_scenario = 'Review fixed + 5 bps'
delayed_schedule = (
    schedule.select('signal_date', 'tsmom_weight', 'market_weight', 'systematic_weight', 'signal_close',
                    pl.col('return_date').shift(-1).alias('return_date'))
    .drop_nulls()
    .join(xle_ohlc.rename({'session_date': 'return_date'}), on='return_date', how='inner')
    .sort('signal_date')
)

timing_rows = []
for strategy in strategies:
    scheduled_history, scheduled_trades = run_execution(schedule, strategy, base_scenario)
    late_history, late_trades = run_execution(delayed_schedule, strategy, base_scenario)
    scheduled = execution_metrics(scheduled_history, scheduled_trades)
    late = execution_metrics(late_history, late_trades)
    timing_rows.append({
        'strategy': strategy, 'scheduled_cagr_pct': scheduled['cagr_pct'],
        'one_session_late_cagr_pct': late['cagr_pct'],
        'cagr_change_pp': late['cagr_pct'] - scheduled['cagr_pct'],
        'scheduled_final_capital': scheduled['final_capital'],
        'late_final_capital': late['final_capital'],
    })
timing_review = pl.DataFrame(timing_rows).with_columns(pl.all().exclude('strategy').round(3))

accounting_audit = pl.DataFrame([{
    'schedule_rows': schedule.height,
    'signal_date_not_before_return_date': int((schedule['signal_date'] >= schedule['return_date']).sum()),
    'null_executable_prices': sum(schedule.select('open', 'close', 'signal_close').null_count().row(0)),
    'nonpositive_prices': int(((schedule['open'] <= 0) | (schedule['close'] <= 0) | (schedule['signal_close'] <= 0)).sum()),
    'dividend_events': xle_ohlc.filter(pl.col('dividend_amount').fill_null(0) != 0).height,
    'split_events': xle_ohlc.filter(pl.col('split_coefficient').fill_null(1) != 1).height,
}])
timing_review, accounting_audit

## <b>Incertidumbre descriptiva</b>

Un block bootstrap circular de 21 sesiones conserva dependencia local aproximada. Estima la incertidumbre de los diferenciales históricos observados, pero no corrige que XLE fue seleccionada después de examinar el mismo historial.

In [ ]:
returns = (
    state['reference_namespace']['strategy_returns']
    .filter(pl.col('asset') == 'XLE')
    .select('return_date', 'alternative', 'strategy_return')
    .pivot(on='alternative', index='return_date', values='strategy_return')
    .drop_nulls()
    .sort('return_date')
)
rng = np.random.default_rng(126)

def block_bootstrap(values, block_length=21, repetitions=2_000):
    samples, n = [], len(values)
    for _ in range(repetitions):
        index = []
        while len(index) < n:
            start = rng.integers(0, n)
            index.extend((start + np.arange(block_length)) % n)
        samples.append(values[np.asarray(index[:n])].mean() * 252 * 100)
    return np.asarray(samples)

bootstrap_rows = []
for benchmark in ('TSMOM total', 'Market momentum', 'Buy & Hold'):
    difference = (returns['Systematic momentum'] - returns[benchmark]).to_numpy()
    sampled = block_bootstrap(difference)
    bootstrap_rows.append({
        'benchmark': benchmark,
        'annualized_log_difference_pct': difference.mean() * 252 * 100,
        'bootstrap_p05_pct': np.quantile(sampled, 0.05),
        'bootstrap_median_pct': np.quantile(sampled, 0.50),
        'bootstrap_p95_pct': np.quantile(sampled, 0.95),
        'probability_positive': (sampled > 0).mean(),
    })
bootstrap_review = pl.DataFrame(bootstrap_rows).with_columns(pl.all().exclude('benchmark').round(3))
bootstrap_review

## <b>Hallazgos operativos antes de la observación en paper</b>

- La identidad de los pesos debe continuar siendo exacta frente a la notebook congelada en cada ejecución.
- La simulación histórica dimensiona inicialmente con el cierre de señal, pero reduce una compra con la apertura observada si no resulta financiable. Ese ajuste conoce el precio de fill y no puede convertirse directamente en una orden preabierta.
- Los precios de apertura históricos no garantizan un fill paper o live. La modalidad Market-on-Open debe observarse por separado y compararse contra el precio registrado.
- Dividendos y el split se contabilizan con las fechas y coeficientes del proveedor; faltan fecha de pago, impuestos y confirmaciones del broker.
- Gold no registra una marca de publicación/cierre definitivo del proveedor. Antes de toda observación hay que exigir un corte diario documentado, datos completos y validación del contrato/cash/órdenes abiertas.